# NAM 基线模型对比 - 完整工作流

**仓库**: https://github.com/yaoyuanArtemis/HKU-NAM-7404

本 Notebook 包含完整的实验流程：
1. 从 GitHub 克隆/更新代码
2. 下载 NAM 论文数据集
3. 运行基线模型对比实验
4. 查看和可视化结果

## 步骤 1: 克隆项目（首次运行）

In [ ]:
import os

# GitHub 配置
GITHUB_REPO = 'https://github.com/yaoyuanArtemis/HKU-NAM-7404.git'
REPO_DIR = '/content/HKU-NAM-7404'
PROJECT_DIR = '/content/HKU-NAM-7404/neural_additive_models'

# 克隆或更新项目
if os.path.exists(REPO_DIR):
    print(f"✓ 项目已存在，更新代码...")
    os.chdir(PROJECT_DIR)
    !git pull origin master
else:
    print("正在从 GitHub 克隆项目...")
    !git clone {GITHUB_REPO} {REPO_DIR}
    os.chdir(PROJECT_DIR)
    print(f"✓ 项目已克隆到: {REPO_DIR}")

print(f"\n当前目录: {os.getcwd()}")
print("\n项目文件:")
!ls -lh *.py | head -10

## 步骤 2: 安装依赖

In [ ]:
!pip install -q xgboost scikit-learn interpret pandas numpy matplotlib
print("✓ 依赖安装完成")

## 步骤 3: 下载 NAM 论文数据集 ⭐

**重要**: 运行此步骤下载数据集（自动从公开来源）

In [ ]:
# 下载数据集
!python download_datasets.py

### 查看下载的数据集

In [ ]:
# 列出已下载的数据集
!ls -lh datasets/

## 步骤 4: 运行实验

### 方案 A: 单数据集快速测试

In [ ]:
# 在 Breast Cancer 数据集上快速测试
!python run_experiment.py \
    --data_path datasets/breast_cancer.csv \
    --target_column target \
    --task classification

### 方案 B: 批量运行所有数据集 ⭐ 推荐

In [ ]:
# 批量运行所有已下载的数据集
# 注意：这会需要 10-30 分钟
!python run_all_datasets.py

## 步骤 5: 查看结果

### 查看单个数据集结果

In [ ]:
import pandas as pd
import glob

# 查找结果文件
result_files = glob.glob('comparison_results/*_comparison.csv')

if result_files:
    latest_result = sorted(result_files)[-1]
    print(f"读取结果: {latest_result}\n")
    
    results_df = pd.read_csv(latest_result)
    print("="*70)
    print("模型对比结果")
    print("="*70)
    print(results_df.to_string(index=False))
else:
    print("未找到结果文件，请先运行实验")

### 查看所有数据集汇总

In [ ]:
# 查看批量实验汇总结果
summary_file = 'all_results/ALL_DATASETS_SUMMARY.csv'

if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file)
    print("="*70)
    print("所有数据集汇总结果")
    print("="*70)
    print(summary_df.to_string(index=False))
else:
    print("未找到汇总结果，请先运行: python run_all_datasets.py")

## 步骤 6: 可视化结果

In [ ]:
import matplotlib.pyplot as plt

if result_files:
    results_df = pd.read_csv(latest_result)
    
    # 性能对比
    test_metric_cols = [col for col in results_df.columns if 'Test' in col and ('AUROC' in col or 'RMSE' in col)]
    
    if test_metric_cols:
        metric_col = test_metric_cols[0]
        
        plt.figure(figsize=(10, 6))
        plt.barh(results_df['Model'], results_df[metric_col])
        plt.xlabel(metric_col)
        plt.title('模型性能对比')
        plt.tight_layout()
        plt.show()
        
        # 训练时间对比
        if 'Train Time (s)' in results_df.columns:
            plt.figure(figsize=(10, 6))
            plt.barh(results_df['Model'], results_df['Train Time (s)'])
            plt.xlabel('训练时间 (秒)')
            plt.title('模型训练时间对比')
            plt.tight_layout()
            plt.show()

### 按数据集可视化（批量实验）

In [ ]:
if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file)
    
    # 找到性能指标列
    metric_cols = [col for col in summary_df.columns if 'Test' in col and ('AUROC' in col or 'RMSE' in col)]
    
    if metric_cols:
        metric_col = metric_cols[0]
        
        # 每个数据集的最佳模型
        plt.figure(figsize=(12, 6))
        
        datasets = summary_df['Dataset'].unique()
        for dataset in datasets:
            dataset_df = summary_df[summary_df['Dataset'] == dataset]
            plt.plot(dataset_df['Model'], dataset_df[metric_col], marker='o', label=dataset)
        
        plt.xlabel('Model')
        plt.ylabel(metric_col)
        plt.title('各数据集上的模型性能对比')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 步骤 7: 下载结果

In [ ]:
from google.colab import files

# 打包所有结果
!zip -r results.zip comparison_results/ all_results/ 2>/dev/null || echo "部分目录不存在"

# 下载
files.download('results.zip')

print("✓ 结果已打包并开始下载")

---

## 📚 NAM 论文数据集说明

### ✅ 可自动下载（6个）

1. **Breast Cancer** - sklearn 内置
2. **California Housing** - sklearn 内置
3. **Adult Income** - UCI ML Repository
4. **Heart Disease** - UCI ML Repository
5. **Telco Churn** - IBM 公开数据
6. **COMPAS Recidivism** - ProPublica 公开数据

### ⚠️ 需要手动下载（2个）

7. **Credit Card Fraud**
   - 链接: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
   - 下载后放到 `datasets/creditcard.csv`

8. **FICO Score**
   - 链接: https://community.fico.com/s/explainable-machine-learning-challenge
   - 需要注册

### ❌ 需要授权（1个）

9. **MIMIC-II**
   - 链接: https://mimic.mit.edu/
   - 需要 PhysioNet 认证

---

## 🔄 更新代码工作流

### 本地修改代码后

```bash
cd /Users/sh01679ml/Desktop/workding-code/google-research/neural_additive_models
git add .
git commit -m "更新"
git push origin master
```

### 在 Colab 中更新

重新运行 **步骤 1** 即可（自动执行 `git pull`）

## 🚀 快捷方式：一键运行全流程

In [ ]:
# 一键：更新代码 + 下载数据 + 运行实验
import os

print("1️⃣ 更新代码...")
os.chdir('/content/HKU-NAM-7404/neural_additive_models')
!git pull origin master

print("\n2️⃣ 下载数据集...")
!python download_datasets.py

print("\n3️⃣ 运行快速测试...")
!python run_experiment.py \
    --data_path datasets/breast_cancer.csv \
    --target_column target \
    --task classification

print("\n✅ 完成！")